# Spanner Export (csv) -> AlloyDB Import (csv)

This notebook provides a quick example of exporting [TheLook public dataset](https://console.cloud.google.com/marketplace/product/bigquery-public-data/thelook-ecommerce) data from a Spanner instance in a non-default VPC using csv format and importing it into AlloyDB. It stitches together incomplete examples from the following sources:
- [Spanner Export to CSV](https://cloud.google.com/spanner/docs/import-export-csv)
- [Google-provided Dataflow Templates](https://cloud.google.com/dataflow/docs/guides/templates/provided-templates)
- [Spanner to Cloud Storage Text Template](https://cloud.google.com/dataflow/docs/guides/templates/provided/cloud-spanner-to-cloud-storage)
- [gcloud dataflow jobs run](https://cloud.google.com/sdk/gcloud/reference/dataflow/jobs/run)
- [Specify a Network for a Dataflow Job](https://cloud.google.com/dataflow/docs/guides/specifying-networks)
- [Import a CSV File to AlloyDB](https://cloud.google.com/alloydb/docs/import-csv-file)

## Source Spanner Schema

This sample assumes the following source schema in Spanner using the GoogleSQL dialect. The target PostgreSQL schema is defined later in the notebok.

```
CREATE TABLE distribution_centers (
  id INT64 NOT NULL,
  name STRING(255),
  latitude NUMERIC,
  longitude NUMERIC,
  distribution_center_geom STRING(MAX),
) PRIMARY KEY(id);

CREATE TABLE events (
  id INT64 NOT NULL,
  user_id INT64,
  sequence_number INT64,
  session_id STRING(255),
  created_at TIMESTAMP,
  ip_address STRING(45),
  city STRING(255),
  state STRING(255),
  postal_code STRING(20),
  browser STRING(255),
  traffic_source STRING(255),
  uri STRING(255),
  event_type STRING(255),
) PRIMARY KEY(id);

CREATE INDEX fk_events_user_3 ON events(user_id);

CREATE TABLE inventory_items (
  id INT64 NOT NULL,
  product_id INT64,
  created_at TIMESTAMP,
  sold_at TIMESTAMP,
  cost NUMERIC,
  product_category STRING(255),
  product_name STRING(255),
  product_brand STRING(255),
  product_retail_price NUMERIC,
  product_department STRING(255),
  product_sku STRING(255),
  product_distribution_center_id INT64,
  CONSTRAINT fk_inventory_items_distribution_center FOREIGN KEY(product_distribution_center_id) REFERENCES distribution_centers(id) ON DELETE NO ACTION,
) PRIMARY KEY(id);

CREATE INDEX fk_inventory_items_distribution_center_8 ON inventory_items(product_distribution_center_id);

CREATE INDEX fk_inventory_items_product_7 ON inventory_items(product_id);

CREATE TABLE order_items (
  id INT64 NOT NULL,
  order_id INT64,
  user_id INT64,
  product_id INT64,
  inventory_item_id INT64,
  status STRING(255),
  created_at TIMESTAMP,
  shipped_at TIMESTAMP,
  delivered_at TIMESTAMP,
  returned_at TIMESTAMP,
  sale_price NUMERIC,
  CONSTRAINT fk_order_items_inventory_item FOREIGN KEY(inventory_item_id) REFERENCES inventory_items(id) ON DELETE NO ACTION,
) PRIMARY KEY(id);

CREATE INDEX fk_order_items_inventory_item_17 ON order_items(inventory_item_id);

CREATE INDEX fk_order_items_order_14 ON order_items(order_id);

CREATE INDEX fk_order_items_product_16 ON order_items(product_id);

CREATE INDEX fk_order_items_user_15 ON order_items(user_id);

CREATE TABLE orders (
  order_id INT64 NOT NULL,
  user_id INT64,
  status STRING(255),
  gender STRING(255),
  created_at TIMESTAMP,
  returned_at TIMESTAMP,
  shipped_at TIMESTAMP,
  delivered_at TIMESTAMP,
  num_of_item INT64,
) PRIMARY KEY(order_id);

ALTER TABLE order_items ADD CONSTRAINT fk_order_items_order FOREIGN KEY(order_id) REFERENCES orders(order_id) ON DELETE NO ACTION;

CREATE INDEX fk_orders_user_20 ON orders(user_id);

CREATE TABLE products (
  id INT64 NOT NULL,
  cost NUMERIC,
  category STRING(255),
  name STRING(255),
  brand STRING(255),
  retail_price NUMERIC,
  department STRING(255),
  sku STRING(255),
  distribution_center_id INT64,
  embedding ARRAY<FLOAT64>,
  embedding_model_version STRING(256),
  CONSTRAINT fk_products_distribution_center FOREIGN KEY(distribution_center_id) REFERENCES distribution_centers(id) ON DELETE NO ACTION,
) PRIMARY KEY(id);

ALTER TABLE inventory_items ADD CONSTRAINT fk_inventory_items_product FOREIGN KEY(product_id) REFERENCES products(id) ON DELETE NO ACTION;

ALTER TABLE order_items ADD CONSTRAINT fk_order_items_product FOREIGN KEY(product_id) REFERENCES products(id) ON DELETE NO ACTION;

CREATE INDEX fk_products_distribution_center_23 ON products(distribution_center_id);

CREATE TABLE users (
  id INT64 NOT NULL,
  first_name STRING(255),
  last_name STRING(255),
  email STRING(255),
  age INT64,
  gender STRING(255),
  state STRING(255),
  street_address STRING(255),
  postal_code STRING(20),
  city STRING(255),
  country STRING(255),
  latitude NUMERIC,
  longitude NUMERIC,
  traffic_source STRING(255),
  created_at TIMESTAMP,
  user_geom STRING(MAX),
) PRIMARY KEY(id);

ALTER TABLE events ADD CONSTRAINT fk_events_user FOREIGN KEY(user_id) REFERENCES users(id) ON DELETE NO ACTION;

ALTER TABLE order_items ADD CONSTRAINT fk_order_items_user FOREIGN KEY(user_id) REFERENCES users(id) ON DELETE NO ACTION;

ALTER TABLE orders ADD CONSTRAINT fk_orders_user FOREIGN KEY(user_id) REFERENCES users(id) ON DELETE NO ACTION;
```

## Basic Setup

### Define Notebook Variables

In [ ]:
project_id = "your-project"  # @param {type:"string"}
region = "your-region"  # @param {type:"string"}
vpc = "your-vpc"  # @param {type:"string"}
source_spanner_instance_id = "spanner-instance"  # @param {type:"string"}
source_spanner_database_id = "ecom-database"  # @param {type:"string"}
export_staging_directory = "gs://your-bucket/staging"  # @param {type:"string"}
export_output_directory = "gs://your-bucket/output"  # @param {type:"string"}
target_alloydb_cluster = "your-alloydb-cluster"  # @param {type:"string"}
target_alloydb_instance = "your-alloydb-instance"  # @param {type:"string"}
target_alloydb_database = "your-database"  # @param {type:"string"}
alloydb_password = input("Please provide a password to be used for 'postgres' database user: ")


### Connect Your Google Cloud Project

In [ ]:
# Configure gcloud.
!gcloud config set project {project_id}

### Configure Logging

In [ ]:
import logging
import sys

# Configure the root logger to output messages with INFO level or above
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

### Add Required Permissions

The default Compute Engine service account requires the permissions defined below to export the database to GCS.

In [ ]:
project_number = ! gcloud projects describe {project_id} --format='value(projectNumber)'
project_number = project_number[0]

roles_array = [
    "roles/spanner.viewer",
    "roles/dataflow.worker",
    "roles/storage.admin",
    "roles/spanner.databaseReader",
    "roles/spanner.databaseAdmin",
    "roles/alloydb.client",
    "roles/serviceusage.serviceUsageConsumer",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:{project_number}-compute@developer.gserviceaccount.com" \
      --role="{r}"


## Define Helper Functions

#### rest_api_helper()

In [ ]:
import requests
import google.auth
import json

# Get an access token based upon the current user
creds, _ = google.auth.default()
authed_session = google.auth.transport.requests.AuthorizedSession(creds)
access_token=creds.token

if project_id:
  authed_session.headers.update({"x-goog-user-project": project_id}) # Required to workaround a project quota bug

def rest_api_helper(
    session: requests.Session,
    url: str,
    http_verb: str,
    request_body: dict = None,
    params: dict = None
  ) -> dict:
  """Calls a REST API using a pre-authenticated requests Session."""

  headers = {"Content-Type": "application/json"}

  try:

    if http_verb == "GET":
      response = session.get(url, headers=headers, params=params)
    elif http_verb == "POST":
      response = session.post(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PUT":
      response = session.put(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PATCH":
      response = session.patch(url, json=request_body, headers=headers, params=params)
    elif http_verb == "DELETE":
      response = session.delete(url, headers=headers, params=params)
    else:
      raise ValueError(f"Unknown HTTP verb: {http_verb}")

    # Raise an exception for bad status codes (4xx or 5xx)
    response.raise_for_status()

    # Check if response has content before trying to parse JSON
    if response.content:
        return response.json()
    else:
        return {} # Return empty dict for empty responses (like 204 No Content)

  except requests.exceptions.RequestException as e:
      # Catch potential requests library errors (network, timeout, etc.)
      # Log detailed error information
      print(f"Request failed: {e}")
      if e.response is not None:
          print(f"Request URL: {e.request.url}")
          print(f"Request Headers: {e.request.headers}")
          print(f"Request Body: {e.request.body}")
          print(f"Response Status: {e.response.status_code}")
          print(f"Response Text: {e.response.text}")
          # Re-raise a more specific error or a custom one
          raise RuntimeError(f"API call failed with status {e.response.status_code}: {e.response.text}") from e
      else:
          raise RuntimeError(f"API call failed: {e}") from e
  except json.JSONDecodeError as e:
      print(f"Failed to decode JSON response: {e}")
      print(f"Response Text: {response.text}")
      raise RuntimeError(f"Invalid JSON received from API: {response.text}") from e



#### wait_for_dataflow_job()

In [ ]:
import time

def wait_for_dataflow_job(job_id: str):
  # Check status of Dataflow job
  job_state = ! gcloud dataflow jobs describe {job_id} --region={region} --format='value(currentState)'

  # Wait until Dataflow job is complete, checking status every 10 seconds
  while job_state[0] in ['JOB_STATE_RUNNING', 'JOB_STATE_PENDING']:
    print(f"Dataflow job {job_id} is in state: {job_state[0]}")
    time.sleep(10)
    job_state = ! gcloud dataflow jobs describe {job_id} --region={region} --format='value(currentState)'

  # Show final Dataflow job state
  print(f"Dataflow job {job_id} final state: {job_state[0]}")
  return job_state[0]

#### run_query()

In [ ]:
# Create AlloyDB Query Helper Function
from sqlalchemy import text, exc
import pandas as pd

async def run_query(pool, sql, output_as_df = True):
  """Executes a SQL query against the AlloyDB database.

  This function accepts a SQL string and performs the following actions:
  - If the SQL statement starts with 'SELECT' or 'WITH' (case-insensitive),
    it executes the query and returns the results as a Pandas DataFrame
    with column names derived from the query.
  - For other types of SQL statements (e.g., INSERT, UPDATE, DELETE),
    it executes the query and returns the SQLAlchemy ResultProxy object
    after committing the transaction.

  Args:
    sql: A string containing the SQL query to execute.

  Returns:
    pandas.DataFrame: If the SQL statement is a SELECT or WITH query,
      a DataFrame containing the query results.
    sqlalchemy.engine.result.ResultProxy: If the SQL statement is not a
      SELECT or WITH query, a ResultProxy object representing the
      result of the execution.
    None: If a `sqlalchemy.exc.ProgrammingError` occurs during query execution,
      the error is printed to the console, and None is returned.

  Raises:
    sqlalchemy.exc.ProgrammingError: If there is an issue with the SQL syntax
      or the database operation. The error is caught, printed, and None is
      returned.

  Example Usage:
  >>> # SELECT query
  >>> sql_select = "SELECT ticker, company_name from investments LIMIT 5"
  >>> df_result = await run_query(sql_select)
  >>> print(df_result)
  >>>
  >>> # INSERT query
  >>> sql_insert = "INSERT INTO investments (ticker, company_name) VALUES ('NEW', 'New Company')"
  >>> insert_result = await run_query(sql_insert)
  >>> print(insert_result) # Output will be the ResultProxy object
  """
  async with pool.connect() as conn:
    if sql.strip().lower().startswith('select') or sql.strip().lower().startswith('with'):
      try:
        result = await conn.execute(sqlalchemy.text(sql))
        if output_as_df:
          rows = result.fetchall()
          column_names = result.keys()
          df = pd.DataFrame(rows, columns=column_names)
          return df
        else:
          return result
      except exc.ProgrammingError as e:
        print(e)
    else:
      try:
        result = await conn.execute(
            text(sql)
        )
        await conn.commit()
        operation_type = sql.split()[0].upper()
        row_count = result.rowcount
        if operation_type in ['INSERT', 'UPDATE', 'DELETE']:
          print(f"{operation_type} statement executed successfully. {row_count} row(s) affected.")
        else:
          print(f"{operation_type} statement executed successfully.")
        return result

      except exc.ProgrammingError as e:
        print(e)



### import_csv_to_alloydb()

In [ ]:
# Reference: https://cloud.google.com/alloydb/docs/import-csv-file#rest-v1
#            https://cloud.google.com/alloydb/docs/reference/rest/v1/projects.locations.operations/get

import time

def import_csv_to_alloydb(import_array):

  operations = []

  for table, files in import_array:
    for f in files:
      url = f"https://alloydb.googleapis.com/v1/projects/{project_id}/locations/{region}/clusters/{target_alloydb_cluster}:import"
      request_body = {
        "gcsUri": f"{export_output_directory}/{table}/{f}",
        "database": f"{target_alloydb_database}",
        "user": "postgres",
        "csvImportOptions": {
          "table": f"{table}",
          #"columns": ["COLUMN1", "COLUMN2"],
          #"fieldDelimiter": "FIELD_DELIMITER",
          #"quoteCharacter": "QUOTE_CHARACTER",
          #"escapeCharacter": "ESCAPE_CHARACTER"
        }
      }
      response = rest_api_helper(authed_session, url, 'POST', request_body, {})
      operations.append([table, response['name']])
      print(response)

  for o in operations:
    operation_complete = False
    while operation_complete == False:
      print(f"Operation for table {o[0]} still running: {o[1]}")
      url = f"https://alloydb.googleapis.com/v1/{o[1]}"
      response = rest_api_helper(authed_session, url, 'GET', request_body, {})
      operation_complete = response['done']
      if operation_complete:
        print(f"Import complete for table: {o[0]}. \nResult: {response}")
        continue
      time.sleep(5)

  return "Import operation complete."



## Export Data from Spanner with Dataflow

In [ ]:
tables = [
    'distribution_centers',
    'users',
    'products',
    'inventory_items',
    'orders',
    'order_items',
    'events'
]

job_ids = []

for t in tables:
  # Kick off Dataflow export job
  result = ! gcloud dataflow jobs run export-spanner-{t} \
      --gcs-location gs://dataflow-templates-{region}/latest/Spanner_to_GCS_Text \
      --region {region} \
      --staging-location '{export_staging_directory}/{t}' \
      --network {vpc} \
      --parameters 'spannerProjectId={project_id},spannerDatabaseId={source_spanner_database_id},spannerInstanceId={source_spanner_instance_id},spannerTable={t},textWritePrefix={export_output_directory}/{t}/'

  job_ids.append(result[2][4:])
  print(result)

for i in job_ids:
  wait_for_dataflow_job(i)

## Import Data to AlloyDB

### Connect to the AlloyDB Cluster
You will need a Postgres AlloyDB instance for the following stages of this notebook. Create one now if you have not already created it.

This function will create a connection pool to your AlloyDB instance using the AlloyDB Python connector. The AlloyDB Python connector will automatically create secure connections to your AlloyDB instance using mTLS.

In [ ]:
import asyncpg

import sqlalchemy
from sqlalchemy.ext.asyncio import AsyncEngine, create_async_engine

from google.cloud.alloydb.connector import AsyncConnector, IPTypes

async def init_connection_pool(connector: AsyncConnector, db_name: str = target_alloydb_database, pool_size: int = 5) -> AsyncEngine:
    # initialize Connector object for connections to AlloyDB
    connection_string = f"projects/{project_id}/locations/{region}/clusters/{target_alloydb_cluster}/instances/{target_alloydb_instance}"

    async def getconn() -> asyncpg.Connection:
        conn: asyncpg.Connection = await connector.connect(
            connection_string,
            "asyncpg",
            user="postgres",
            password=alloydb_password,
            db=db_name,
            ip_type=IPTypes.PRIVATE,
        )
        return conn

    pool = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        pool_size=pool_size,
        max_overflow=0,
        isolation_level='AUTOCOMMIT'
    )
    return pool

connector = AsyncConnector()

postgres_db_pool = await init_connection_pool(connector, "postgres")
ecom_db_pool = await init_connection_pool(connector, f"{target_alloydb_database}")

### Define AlloyDB Schema

In [ ]:
# Uncomment and run this cell to drop the existing database if you want to replace it.

# Close existing connections to the database
sql = f"""SELECT pg_terminate_backend(pg_stat_activity.pid)
FROM pg_stat_activity
WHERE pg_stat_activity.datname = '{target_alloydb_database}'
  AND pid <> pg_backend_pid();"""
await run_query(postgres_db_pool, sql)

# Drop the database
sql = f"DROP DATABASE {target_alloydb_database};"
await run_query(postgres_db_pool, sql)

# Reinitiate the connection pool
ecom_db_pool = await init_connection_pool(connector, f"{target_alloydb_database}")

In [ ]:
# Use postgres_db_pool to create the database
sql = f"CREATE DATABASE {target_alloydb_database};"
result = await run_query(postgres_db_pool, sql)

In [ ]:
# Use ecom_db_pool to create the rest of the schema objects
sql_array = []

sql_array.append("CREATE EXTENSION IF NOT EXISTS vector;")

sql_array.append("CREATE EXTENSION IF NOT EXISTS google_ml_integration;")

sql_array.append("""
CREATE TABLE distribution_centers (
  id BIGINT PRIMARY KEY NOT NULL,
  name VARCHAR(255),
  latitude NUMERIC,
  longitude NUMERIC,
  distribution_center_geom TEXT
);""")

sql_array.append("""CREATE TABLE users (
  id BIGINT PRIMARY KEY NOT NULL,
  first_name VARCHAR(255),
  last_name VARCHAR(255),
  email VARCHAR(255),
  age BIGINT,
  gender VARCHAR(255),
  state VARCHAR(255),
  street_address VARCHAR(255),
  postal_code VARCHAR(20),
  city VARCHAR(255),
  country VARCHAR(255),
  latitude NUMERIC,
  longitude NUMERIC,
  traffic_source VARCHAR(255),
  created_at TIMESTAMP,
  user_geom TEXT
);""")

sql_array.append("""CREATE TABLE events (
  id BIGINT PRIMARY KEY NOT NULL,
  user_id BIGINT,
  sequence_number BIGINT,
  session_id VARCHAR(255),
  created_at TIMESTAMP,
  ip_address VARCHAR(45),
  city VARCHAR(255),
  state VARCHAR(255),
  postal_code VARCHAR(20),
  browser VARCHAR(255),
  traffic_source VARCHAR(255),
  uri TEXT,
  event_type VARCHAR(255)
);""")

sql_array.append("""CREATE TABLE products (
  id BIGINT PRIMARY KEY NOT NULL,
  cost NUMERIC,
  category VARCHAR(255),
  name VARCHAR(255),
  brand VARCHAR(255),
  retail_price NUMERIC,
  department VARCHAR(255),
  sku VARCHAR(255),
  distribution_center_id BIGINT,
  embedding vector(768),
  embedding_model_version VARCHAR(256)
);""")

sql_array.append("""CREATE TABLE inventory_items (
  id BIGINT PRIMARY KEY NOT NULL,
  product_id BIGINT,
  created_at TIMESTAMP,
  sold_at TIMESTAMP,
  cost NUMERIC,
  product_category VARCHAR(255),
  product_name VARCHAR(255),
  product_brand VARCHAR(255),
  product_retail_price NUMERIC,
  product_department VARCHAR(255),
  product_sku VARCHAR(255),
  product_distribution_center_id BIGINT
);""")

sql_array.append("""CREATE TABLE orders (
  order_id BIGINT PRIMARY KEY NOT NULL,
  user_id BIGINT,
  status VARCHAR(255),
  gender VARCHAR(255),
  created_at TIMESTAMP,
  returned_at TIMESTAMP,
  shipped_at TIMESTAMP,
  delivered_at TIMESTAMP,
  num_of_item BIGINT
);""")

sql_array.append("""CREATE TABLE order_items (
  id BIGINT PRIMARY KEY NOT NULL,
  order_id BIGINT,
  user_id BIGINT,
  product_id BIGINT,
  inventory_item_id BIGINT,
  status VARCHAR(255),
  created_at TIMESTAMP,
  shipped_at TIMESTAMP,
  delivered_at TIMESTAMP,
  returned_at TIMESTAMP,
  sale_price NUMERIC
);""")

for sql in sql_array:
  result = await run_query(ecom_db_pool, sql)

### Import Data

### Run Import

In [ ]:
# Format [ [ <table>, [<list-of-files>] ] ]
import_array = [
    ['distribution_centers',['-00000-of-00001.csv']],
    ['users',['-00000-of-00001.csv']],
    ['products',['-00000-of-00001.csv']],
    ['orders',['-00000-of-00001.csv']],
    ['events',['-00000-of-00002.csv','-00001-of-00002.csv']],
    ['inventory_items',['-00000-of-00001.csv']],
    ['order_items',['-00000-of-00001.csv']],
]

import_csv_to_alloydb(import_array)

### Validate Row Counts

In [ ]:
sql = """
SELECT 'distribution_centers' AS table_name, (SELECT COUNT(*) FROM distribution_centers) AS actual_row_count, 10 AS target_row_count
UNION ALL
SELECT 'events', (SELECT COUNT(*) FROM events), 2438862
UNION ALL
SELECT 'inventory_items', (SELECT COUNT(*) FROM inventory_items), 494254
UNION ALL
SELECT 'orders', (SELECT COUNT(*) FROM orders), 125905
UNION ALL
SELECT 'order_items', (SELECT COUNT(*) FROM order_items), 182905
UNION ALL
SELECT 'products', (SELECT COUNT(*) FROM products), 29120
UNION ALL
SELECT 'users', (SELECT COUNT(*) FROM users), 100000;
"""

await run_query(ecom_db_pool, sql)

### Create Foreign Keys

Wait until the data is imported to create foreign keys to prevent foreign key violations and to improve performance.

In [ ]:
sql_array = []

# Add FK constraints
sql_array.append("ALTER TABLE events ADD CONSTRAINT fk_events_user FOREIGN KEY(user_id) REFERENCES users(id) ON DELETE NO ACTION;")
sql_array.append("ALTER TABLE products ADD CONSTRAINT fk_products_distribution_center FOREIGN KEY(distribution_center_id) REFERENCES distribution_centers(id) ON DELETE NO ACTION;")
sql_array.append("ALTER TABLE inventory_items ADD CONSTRAINT fk_inventory_items_distribution_center FOREIGN KEY(product_distribution_center_id) REFERENCES distribution_centers(id) ON DELETE NO ACTION;")
sql_array.append("ALTER TABLE inventory_items ADD CONSTRAINT fk_inventory_items_product FOREIGN KEY(product_id) REFERENCES products(id) ON DELETE NO ACTION;")
sql_array.append("ALTER TABLE orders ADD CONSTRAINT fk_orders_user FOREIGN KEY(user_id) REFERENCES users(id) ON DELETE NO ACTION;")
sql_array.append("ALTER TABLE order_items ADD CONSTRAINT fk_order_items_inventory_item FOREIGN KEY(inventory_item_id) REFERENCES inventory_items(id) ON DELETE NO ACTION;")
sql_array.append("ALTER TABLE order_items ADD CONSTRAINT fk_order_items_order FOREIGN KEY(order_id) REFERENCES orders(order_id) ON DELETE NO ACTION;")
sql_array.append("ALTER TABLE order_items ADD CONSTRAINT fk_order_items_product FOREIGN KEY(product_id) REFERENCES products(id) ON DELETE NO ACTION;")
sql_array.append("ALTER TABLE order_items ADD CONSTRAINT fk_order_items_user FOREIGN KEY(user_id) REFERENCES users(id) ON DELETE NO ACTION;")

# Add FK indexes
sql_array.append("CREATE INDEX fk_events_user_3 ON events(user_id);")
sql_array.append("CREATE INDEX fk_products_distribution_center_23 ON products(distribution_center_id);")
sql_array.append("CREATE INDEX fk_inventory_items_distribution_center_8 ON inventory_items(product_distribution_center_id);")
sql_array.append("CREATE INDEX fk_inventory_items_product_7 ON inventory_items(product_id);")
sql_array.append("CREATE INDEX fk_orders_user_20 ON orders(user_id);")
sql_array.append("CREATE INDEX fk_order_items_inventory_item_17 ON order_items(inventory_item_id);")
sql_array.append("CREATE INDEX fk_order_items_order_14 ON order_items(order_id);")
sql_array.append("CREATE INDEX fk_order_items_product_16 ON order_items(product_id);")
sql_array.append("CREATE INDEX fk_order_items_user_15 ON order_items(user_id);")

for sql in sql_array:
  response = await run_query(ecom_db_pool, sql)

### Validate Foreign Keys

In [ ]:
sql_array = []

# Validate FK constraints
sql_array.append("ALTER TABLE events VALIDATE CONSTRAINT fk_events_user;")
sql_array.append("ALTER TABLE products VALIDATE CONSTRAINT fk_products_distribution_center;")
sql_array.append("ALTER TABLE inventory_items VALIDATE CONSTRAINT fk_inventory_items_distribution_center;")
sql_array.append("ALTER TABLE inventory_items VALIDATE CONSTRAINT fk_inventory_items_product;")
sql_array.append("ALTER TABLE orders VALIDATE CONSTRAINT fk_orders_user;")
sql_array.append("ALTER TABLE order_items VALIDATE CONSTRAINT fk_order_items_inventory_item;")
sql_array.append("ALTER TABLE order_items VALIDATE CONSTRAINT fk_order_items_order;")
sql_array.append("ALTER TABLE order_items VALIDATE CONSTRAINT fk_order_items_product;")
sql_array.append("ALTER TABLE order_items VALIDATE CONSTRAINT fk_order_items_user;")

for sql in sql_array:
  response = await run_query(ecom_db_pool, sql)

## Export from AlloyDB to SQL File

You can now optionally export the new AlloyDB tables as a single `.sql` file for easier loading in the future. 

In [ ]:
# Add required permissions

project_number = ! gcloud projects describe {project_id} --format='value(projectNumber)'
project_number = project_number[0]

roles_array = [
    "roles/storage.admin",
]

for r in roles_array:
  ! gcloud projects add-iam-policy-binding {project_id} \
      --member="serviceAccount:service-{project_number}@gcp-sa-alloydb.iam.gserviceaccount.com" \
      --role="{r}"

In [ ]:
url = f"https://alloydb.googleapis.com/v1/projects/{project_id}/locations/{region}/clusters/{target_alloydb_cluster}:export"
request_body = {
  "gcs_destination": {
    "uri": f"{export_output_directory}/alloydb_export/{target_alloydb_database}.sql"
  },
  "database": f"{target_alloydb_database}",
  "sql_export_options": {
    "schema_only": "false",
    "tables": [
        'distribution_centers',
        'users',
        'products',
        'inventory_items',
        'orders',
        'order_items',
        'events'
    ],
    "clean_target_objects": "false",
    "if_exist_target_objects": "false"
  }
}

result = rest_api_helper(authed_session, url, 'POST', request_body, {})
print(f"Kicked off export: {result}")

operation_id = result['name']

operation_complete = False
while operation_complete == False:
  print(f"Export still running: {operation_id}")
  url = f"https://alloydb.googleapis.com/v1/{operation_id}"
  response = rest_api_helper(authed_session, url, 'GET', request_body, {})
  operation_complete = response['done']
  if operation_complete:
    print(f"Operation complete. \nResult: {response}")
    continue
  time.sleep(5)